# Empirical Risk Minimization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Minimizing the error you can measure — and why it is optimistically biased
Empirical risk is the average loss on the sample; ERM returns its minimizer. These five examples show ERM working on 0-1 and squared loss, show empirical risk is an honest estimate for a fixed model, then expose the optimistic bias that makes the training minimum untrustworthy.

In [ ]:
# 1) 0-1 empirical risk of threshold rules: sweep t, ERM picks the minimum
X=np.array([1,2,3,4,5]); y=np.array([0,0,1,1,1])
ts=np.linspace(0.5,5.5,50); R=[zero_one((X>t).astype(int),y) for t in ts]
plt.figure(figsize=(6,3)); plt.plot(ts,R); plt.scatter([2.5],[0],c='r',zorder=5)
plt.xlabel('threshold t'); plt.ylabel('empirical 0-1 risk'); plt.title('ERM picks t=2.5 (risk 0)')
assert min(R)==0.0 and abs(ts[int(np.argmin(R))]-2.5)<0.6

In [ ]:
# 2) squared-loss constant fit: MSE(c) is a parabola minimized at the mean
ys=np.array([2.,4.,6.]); cs=np.linspace(1,7,50); mse=[np.mean((c-ys)**2) for c in cs]
plt.figure(figsize=(6,3)); plt.plot(cs,mse); plt.scatter([4],[np.var(ys)],c='r',zorder=5)
plt.xlabel('constant c'); plt.ylabel('MSE'); plt.title('min at c=mean=4, risk=variance=2.667')
assert abs(cs[int(np.argmin(mse))]-4)<0.15 and abs(np.min(mse)-2.667)<0.05

In [ ]:
# 3) for a FIXED model, empirical risk concentrates on the true risk as m grows
p_true=0.3; ms=[5,20,100,500,2000]; rng=np.random.default_rng(0)
est=[[rng.binomial(m,p_true)/m for _ in range(300)] for m in ms]
means=[np.mean(e) for e in est]; sds=[np.std(e) for e in est]
plt.figure(figsize=(6,3)); plt.errorbar(ms,means,yerr=sds,fmt='-o',capsize=3); plt.axhline(p_true,ls='--',c='r')
plt.xscale('log'); plt.xlabel('sample size m'); plt.ylabel('empirical risk'); plt.title('unbiased for fixed h; variance ~ 1/m')
assert sds[-1]<sds[0] and abs(means[-1]-p_true)<0.03

In [ ]:
# 4) but the MINIMUM training error is optimistically biased, worse for bigger classes
rng=np.random.default_rng(2); m=50; true_err=0.5
best=lambda H: min(rng.binomial(m,true_err)/m for _ in range(H))
Hs=[1,2,10,100,1000]; bias=[np.mean([true_err-best(H) for _ in range(200)]) for H in Hs]
plt.figure(figsize=(6,3)); plt.semilogx(Hs,bias,'-o'); plt.xlabel('|H|'); plt.ylabel('true - min(train error)')
plt.title('bigger class => more optimistic training min')
assert bias[-1]>bias[0]

In [ ]:
# 5) the consequence: polynomial ERM drives train error down while test error turns up
rng=np.random.default_rng(3); xt=np.sort(rng.uniform(-1,1,15)); yt=xt**2+rng.normal(0,0.1,15)
xv=np.linspace(-1,1,100); yv=xv**2; degs=range(1,12); trn=[]; tst=[]
for d in degs:
    c=np.polyfit(xt,yt,d); trn.append(np.mean((np.polyval(c,xt)-yt)**2)); tst.append(np.mean((np.polyval(c,xv)-yv)**2))
plt.figure(figsize=(6,3)); plt.semilogy(list(degs),trn,'-o',ms=3,label='train'); plt.semilogy(list(degs),tst,'-s',ms=3,label='test')
plt.legend(); plt.xlabel('polynomial degree'); plt.ylabel('MSE (log)'); plt.title('train falls, test turns up (overfitting)')
assert trn[-1]<trn[0] and tst[-1]>min(tst)